# Space-map Alignment Example

This notebook demonstrates how to align spatial transcriptomics sections using Space-map.

**What you need:** A CSV file with `x`, `y`, and `layer` columns.

**What you get:** Aligned coordinates + quality metrics (Dice, SSIM, PSNR, MSE).

## 1. Setup

Install space-map if you haven't already:
```bash
pip install space-map
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import space_map
from space_map import Slice
from space_map.flow import FlowImport, AutoFlowMultiCenter4
from space_map.find.err import err_dice, err_ssim, err_psnr, err_mse

print(f"space-map version: {space_map.__version__}")

## 2. Load your data

Your CSV should have at minimum three columns:
- `x` — x coordinate of each cell
- `y` — y coordinate of each cell  
- `layer` — which tissue section this cell belongs to

**Change the path below to your own data file.**

In [ ]:
# ============================================================
# >>> CHANGE THIS to your CSV file path <<<
DATA_PATH = "../examples/cells2.csv.gz"
# ============================================================

# Column names (change if your CSV uses different names)
X_COL = "x"
Y_COL = "y"
LAYER_COL = "layer"

# Load data
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} cells")
print(f"Columns: {list(df.columns)}")
print(f"Layers: {sorted(df[LAYER_COL].unique())}")
df.head()

In [ ]:
# Split into per-layer coordinate arrays
layers = sorted(df[LAYER_COL].unique())
xys = []
for layer in layers:
    sub = df[df[LAYER_COL] == layer]
    xys.append(sub[[X_COL, Y_COL]].values.astype(np.float64))

ids = [str(l) for l in layers]
print(f"{len(layers)} layers, {sum(len(x) for x in xys):,} cells total")

## 3. Visualize raw data (before alignment)

In [ ]:
def plot_layers(xys, ids, title, n_show=4):
    """Show a few layers side by side."""
    n_show = min(n_show, len(xys))
    fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))
    if n_show == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        ax.scatter(xys[i][:, 0], xys[i][:, 1], s=0.1, alpha=0.3)
        ax.set_title(f"Layer {ids[i]}")
        ax.set_aspect("equal")
        ax.invert_yaxis()
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_layers(xys, ids, "Raw data (before alignment)")

## 4. Run Space-map alignment

Two-stage pipeline:
1. **Affine registration** — coarse global alignment (rotation, scaling, translation)
2. **LDDMM registration** — fine non-rigid deformation

In [ ]:
import tempfile, time

# Create temporary project directory
base_path = tempfile.mkdtemp(prefix="spacemap_notebook_")

# Initialize project
flow = FlowImport(base_path)
slices = flow.init_xys(xys, ids)

In [ ]:
# Stage 1: Affine registration
t0 = time.time()
af = AutoFlowMultiCenter4(slices, initJKey=Slice.rawKey)
af.affine(useKey="DF", show=False)
print(f"Affine done in {time.time() - t0:.1f}s")

In [ ]:
# Stage 2: LDDMM non-rigid registration
t1 = time.time()
af.ldm_pair(fromKey=Slice.align1Key, toKey=Slice.align2Key, show=False)
print(f"LDDMM done in {time.time() - t1:.1f}s")
print(f"Total: {time.time() - t0:.1f}s")

In [ ]:
# Extract aligned coordinates
aligned_xys = [s.ps(Slice.align2Key) for s in slices]

## 5. Visualize alignment results

In [ ]:
plot_layers(aligned_xys, ids, "After Space-map alignment")

In [ ]:
def plot_overlay(xy1, xy2, id1, id2, title):
    """Overlay two adjacent layers to visualize alignment."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for ax, (a, b, label) in zip(axes, [
        (xys[id1], xys[id2], "Before alignment"),
        (xy1, xy2, "After alignment")
    ]):
        ax.scatter(a[:, 0], a[:, 1], s=0.1, alpha=0.3, c="blue", label=f"Layer {ids[id1]}")
        ax.scatter(b[:, 0], b[:, 1], s=0.1, alpha=0.3, c="red", label=f"Layer {ids[id2]}")
        ax.set_title(label)
        ax.set_aspect("equal")
        ax.invert_yaxis()
        ax.legend(markerscale=20)
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

# Show overlay for a pair of adjacent layers
pair_idx = len(ids) // 2  # pick a middle pair
plot_overlay(aligned_xys[pair_idx], aligned_xys[pair_idx + 1],
             pair_idx, pair_idx + 1,
             f"Layers {ids[pair_idx]} & {ids[pair_idx + 1]}")

## 6. Compute quality metrics

We compare adjacent layer pairs before and after alignment using four metrics:

| Metric | Range | Better |
|--------|-------|--------|
| **Dice** | 0–1 | Higher |
| **SSIM** | -1–1 | Higher |
| **PSNR** | 0–∞ | Higher |
| **MSE** | 0–∞ | Lower |

In [ ]:
metrics_funcs = {"dice": err_dice, "ssim": err_ssim, "psnr": err_psnr, "mse": err_mse}
records = []

for i in range(len(ids) - 1):
    raw_i = space_map.show_img(xys[i])
    raw_j = space_map.show_img(xys[i + 1])
    ali_i = space_map.show_img(aligned_xys[i])
    ali_j = space_map.show_img(aligned_xys[i + 1])

    row = {"layer_i": ids[i], "layer_j": ids[i + 1]}
    for name, func in metrics_funcs.items():
        row[f"{name}_raw"] = func(raw_i, raw_j)
        row[f"{name}_aligned"] = func(ali_i, ali_j)
    records.append(row)

metrics_df = pd.DataFrame(records)
metrics_df.head(10)

In [ ]:
# Summary
print("=== Alignment Quality (mean ± std) ===")
for m in ["dice", "ssim", "psnr", "mse"]:
    raw = metrics_df[f"{m}_raw"]
    ali = metrics_df[f"{m}_aligned"]
    print(f"  {m.upper():5s}:  {raw.mean():.4f}±{raw.std():.4f}  ->  {ali.mean():.4f}±{ali.std():.4f}")

In [ ]:
# Plot metrics comparison
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, m in zip(axes, ["dice", "ssim", "psnr", "mse"]):
    x = range(len(metrics_df))
    ax.plot(x, metrics_df[f"{m}_raw"], "o-", label="Raw", alpha=0.7, markersize=3)
    ax.plot(x, metrics_df[f"{m}_aligned"], "s-", label="Aligned", alpha=0.7, markersize=3)
    ax.set_title(m.upper())
    ax.set_xlabel("Layer pair")
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Save results

In [ ]:
import os

output_dir = "results"
os.makedirs(output_dir, exist_ok=True)

# Save aligned coordinates
aligned_df = df.copy()
for layer, xy in zip(layers, aligned_xys):
    mask = aligned_df[LAYER_COL] == layer
    aligned_df.loc[mask, X_COL] = xy[:, 0]
    aligned_df.loc[mask, Y_COL] = xy[:, 1]
aligned_df.to_csv(os.path.join(output_dir, "aligned.csv.gz"), index=False)

# Save metrics
metrics_df.to_csv(os.path.join(output_dir, "metrics.csv"), index=False)

print(f"Saved to {output_dir}/")
print(f"  aligned.csv.gz  - aligned coordinates")
print(f"  metrics.csv     - per-layer-pair metrics")